In [3]:

import numpy as np

print("====================================")
print(" FERTILIZER OPTIMIZATION USING GWO")
print("====================================")

# 1. User inputs
crop = input("Enter crop (Rice/Wheat/Maize/Cotton): ").lower()
area = float(input("Enter field area in acres: "))

season = input("Enter season (Summer/Winter/Rainy): ").lower()
soil = input("Enter soil type (Sandy/Clay/Loamy): ").lower()
irrigation = input("Enter irrigation (Rainfed/Irrigated): ").lower()

# 2. Crop nutrient requirements (kg/hectare)
crop_data = {
    "rice": [120, 60, 40],
    "wheat": [120, 60, 40],
    "maize": [150, 75, 60],
    "cotton": [120, 60, 60]
}

if crop not in crop_data:
    print("Invalid crop name!")
    raise SystemExit

# Base nutrient requirements
N, P, K = crop_data[crop]

# 3. Season adjustment (illustrative factors)
season_factor = {
    "summer": 1.05,
    "winter": 0.95,
    "rainy": 1.00
}

if season not in season_factor:
    print("Invalid season!")
    raise SystemExit

factor = season_factor[season]

# Convert area into hectares
area_ha = area * 0.404686

# Adjust nutrient targets
target = np.array([N, P, K]) * factor

# 4. GWO parameters
wolves = 25
iterations = 100
dim = 3

lower = np.zeros(3)
upper = target * 1.5

# Cost per kg of nutrient (illustrative)
cost = np.array([50, 40, 35])

# 5. Objective function
def objective(x):
    total_cost = np.sum(x * cost)

    # Ensure nutrient requirements are met
    penalty = np.sum(
        np.maximum(target - x, 0) ** 2
    )

    return total_cost + 1000 * penalty

# 6. Initialize wolves
population = np.random.uniform(
    lower, upper, (wolves, dim)
)

alpha = np.zeros(dim)
beta = np.zeros(dim)
delta = np.zeros(dim)

alpha_score = beta_score = delta_score = float("inf")

# 7. Grey Wolf Optimization
for t in range(iterations):

    for i in range(wolves):
        fitness = objective(population[i])

        if fitness < alpha_score:
            delta_score, delta = beta_score, beta.copy()
            beta_score, beta = alpha_score, alpha.copy()
            alpha_score, alpha = fitness, population[i].copy()

        elif fitness < beta_score:
            delta_score, delta = beta_score, beta.copy()
            beta_score, beta = fitness, population[i].copy()

        elif fitness < delta_score:
            delta_score = fitness
            delta = population[i].copy()

    a = 2 - 2 * t / iterations

    for i in range(wolves):
        X = []

        for leader in [alpha, beta, delta]:
            r1 = np.random.rand(dim)
            r2 = np.random.rand(dim)

            A = 2 * a * r1 - a
            C = 2 * r2

            D = np.abs(C * leader - population[i])
            X.append(leader - A * D)

        population[i] = np.mean(X, axis=0)
        population[i] = np.clip(
            population[i], lower, upper
        )

# 8. Final output
recommended = alpha * area_ha

print("\n====================================")
print(" OPTIMAL FERTILIZER RECOMMENDATION")
print("====================================")

print("Crop:", crop.capitalize())
print("Field Area:", area, "acres")
print("Season:", season.capitalize())
print("Soil Type:", soil.capitalize())
print("Irrigation:", irrigation.capitalize())

print("\nRecommended Nutrients for Entire Field:")
print("Nitrogen (N):", round(recommended[0], 2), "kg")
print("Phosphorus (P):", round(recommended[1], 2), "kg")
print("Potassium (K):", round(recommended[2], 2), "kg")

total_cost = np.sum(recommended * cost)

print("\nEstimated Total Cost: Rs.",
      round(total_cost, 2))

print("\nOptimization completed using GWO.")
print("====================================")


 FERTILIZER OPTIMIZATION USING GWO
Enter crop (Rice/Wheat/Maize/Cotton): Rice
Enter field area in acres: 3
Enter season (Summer/Winter/Rainy): Summer
Enter soil type (Sandy/Clay/Loamy): Clay
Enter irrigation (Rainfed/Irrigated): Irrigated

 OPTIMAL FERTILIZER RECOMMENDATION
Crop: Rice
Field Area: 3.0 acres
Season: Summer
Soil Type: Clay
Irrigation: Irrigated

Recommended Nutrients for Entire Field:
Nitrogen (N): 152.85 kg
Phosphorus (P): 76.51 kg
Potassium (K): 51.37 kg

Estimated Total Cost: Rs. 12500.81

Optimization completed using GWO.
